# Analyse de données sur le Covid-19 avec Python, NumPy et Pandas

Notebook de l'article [Analyse de données sur le Covid-19 avec Python, NumPy et Pandas](https://moncoachdata.com/blog/analyse-de-donnees-sur-le-covid-19/) (MonCoachData).

Exécute les cellules **dans l'ordre**. Le code est testé avec Python 3.13 et Pandas 3. Les données sont les fichiers de séries temporelles du jeu [Novel Corona Virus 2019 Dataset](https://www.kaggle.com/datasets/sudalairajkumar/novel-corona-virus-2019-dataset), dans leur version du **14 juin 2020**, copiés dans le dossier de ce notebook sur GitHub. Source : [CSSE, université Johns Hopkins](https://github.com/CSSEGISandData/COVID-19), licence CC BY 4.0.

## Importation des bibliothèques

In [ ]:
import pandas as pd
import numpy as np

## Lecture des données

Les trois fichiers sont lus directement depuis GitHub (versions du 14 juin 2020).

In [ ]:
dataset_url = "https://raw.githubusercontent.com/moncoachdata/blog-data-science/main/analyse-de-donnees-sur-le-covid-19/"

df_conf      = pd.read_csv(dataset_url + "time_series_covid_19_confirmed.csv")
df_death     = pd.read_csv(dataset_url + "time_series_covid_19_deaths.csv")
df_recovered = pd.read_csv(dataset_url + "time_series_covid_19_recovered.csv")

print(df_conf.shape)
print(df_death.shape)
print(df_recovered.shape)

In [ ]:
df_conf

## Tri et observation des DataFrames

In [ ]:
df_conf = df_conf.sort_values(
    by=['Province/State','Country/Region'])
df_conf = df_conf.reset_index(drop=True)

df_death = df_death.sort_values(
    by=['Province/State','Country/Region'])
df_death = df_death.reset_index(drop=True)

df_recovered = df_recovered.sort_values(
    by=['Province/State','Country/Region'])
df_recovered = df_recovered.reset_index(drop=True)

In [ ]:
df_conf

Comparons les lignes des trois DataFrames.

In [ ]:
# Vérifie si des colonnes de 2 dataframes sont les mêmes
print(df_conf[["Province/State","Country/Region"]].equals(
      df_death[["Province/State","Country/Region"]]))

In [ ]:
print(df_conf[["Province/State","Country/Region"]].equals(
      df_recovered[["Province/State","Country/Region"]]))

Le Canada est découpé par province dans `df_conf` et `df_death`, mais il occupe une seule ligne dans `df_recovered`.

In [ ]:
print(df_conf.loc[df_conf["Country/Region"] == "Canada", "Province/State"].tolist())
print(df_recovered.loc[df_recovered["Country/Region"] == "Canada", "Province/State"].tolist())

## Extraire les noms des colonnes contenant des dates

In [ ]:
dates_conf = df_conf.columns[4:]
dates_death = df_death.columns[4:]
print(dates_conf)
print(dates_death)

# s'assurer que les dates sont les mêmes pour les 2 DataFrames
if dates_conf.equals(dates_death):
    print("Les dates sont les mêmes")

## « Dépivoter » les DataFrames avec `melt()`

In [ ]:
df_conf_melted = df_conf.melt(id_vars=['Province/State',
                              'Country/Region', 'Lat', 'Long'],
                              value_vars=dates_conf,
                              var_name='Date',
                              value_name='Confirmed')

df_death_melted = df_death.melt(id_vars=['Province/State',
                                'Country/Region', 'Lat', 'Long'],
                                value_vars=dates_death,
                                var_name='Date',
                                value_name='Deaths')

In [ ]:
df_conf_melted

In [ ]:
df_death_melted

In [ ]:
print(df_conf_melted.shape)
print(df_death_melted.shape)

## Combiner les deux DataFrames

In [ ]:
df_combined = pd.concat([df_conf_melted,
                         df_death_melted["Deaths"]],
                         axis = 1,
                         sort = False)

df_combined

Pourcentage de décès par rapport aux cas confirmés.

In [ ]:
df_combined["Percentage"] = df_combined["Deaths"] / df_combined["Confirmed"]

df_combined

Les `NaN` viennent de la division de 0 par 0 : on les remplace par 0.

In [ ]:
df_combined["Percentage"] = df_combined["Percentage"].fillna(0)

df_combined

Types des colonnes : depuis Pandas 3, le texte a le type `str`.

In [ ]:
df_combined.dtypes

## Convertir la colonne `Date`

In [ ]:
import datetime

def format_time(datetime_str):
    d = datetime.datetime.strptime(datetime_str, '%m/%d/%y')
    return d.date()

df_combined["Date"] = df_combined["Date"].apply(format_time)

df_combined

## Nombre quotidien de cas confirmés et de décès

La sortie est longue (une table par jour) : tu peux interrompre la cellule après quelques secondes.

In [ ]:
grouped_by_date = df_combined.groupby("Date")

for date, group in grouped_by_date:
    print("==========")
    print(date)
    print("==========")
    print(group[["Province/State","Country/Region", "Confirmed",
        "Deaths"]])

## Total quotidien par pays

In [ ]:
df_daily = df_combined.groupby(["Date","Country/Region"]).aggregate(
    {'Confirmed': 'sum', 'Deaths': 'sum'})

df_daily

## Données d'un jour précis

In [ ]:
df_day = df_daily.loc[format_time('3/22/20')]

df_day

### Un pays et un jour précis

In [ ]:
df_day_usa = df_daily.loc[(format_time('5/22/20'),'US')]

df_day_usa

In [ ]:
df_day_france = df_daily.loc[(format_time('5/22/20'),'France')]

df_day_france

## Données du jour le plus récent

In [ ]:
# Obtenir la date la plus récente
most_recent_date = df_combined['Date'].max()
print(most_recent_date)

In [ ]:
# Obtenir toutes les données pour la date la plus récente
df_most_recent = df_daily.loc[most_recent_date,:]

df_most_recent

## Les 10 premiers pays en cas confirmés

In [ ]:
df_most_confirmed_recent_sorted = df_most_recent.sort_values(
    by="Confirmed", ascending=False)

df_most_confirmed_recent_sorted.head(10)

## Chiffres les plus récents pour un pays (saisis le nom en anglais, par exemple `France`)

In [ ]:
df_most_recent[df_most_recent.index == input("Entrer le nom de pays : ")]

## Les 10 pays avec le plus de décès

In [ ]:
df_most_deaths_recent_sorted = df_most_recent.sort_values(
    by="Deaths", ascending=False)

df_most_deaths_recent_sorted = df_most_deaths_recent_sorted[
    df_most_deaths_recent_sorted.Deaths > 0]

df_most_deaths_recent_sorted.head(10)

## Chine continentale

In [ ]:
# obtenir toutes les lignes pour la Chine continentale
df_combined_china = df_combined[df_combined['Country/Region'] == "China"]

# en utilisant le dataframe de la chine, grouper par date et par province
df_combined_china_date = df_combined_china.groupby(["Date","Province/State"]).aggregate(
    {'Confirmed': 'sum', 'Deaths': 'sum'})

# trouver les données pour la date la plus récente
df_combined_china_most_recent = df_combined_china_date.loc[most_recent_date,:]

df_combined_china_most_recent

In [ ]:
df_combined_china_most_recent_sorted = df_combined_china_most_recent.sort_values(
    by="Confirmed", ascending=False).head(10)

df_combined_china_most_recent_sorted

# Visualisation des données du Covid-19

## Diagrammes à barres

Le filtre `pays` limite le tracé à quatre pays : retire la condition `if` pour obtenir un graphique par pays.

In [ ]:
import matplotlib.pyplot as plt

plt.style.use('ggplot')
plt.rcParams["figure.figsize"] = (12, 7)

# pays à tracer (retire ce filtre pour obtenir un graphique par pays)
pays = ["Afghanistan", "US", "Singapore", "France"]

for country, df_country in df_daily.groupby(level=1):
    if country not in pays:
        continue

    dates = list(df_country.index.get_level_values('Date'))
    confirmed = list(df_country.Confirmed)
    deaths = list(df_country.Deaths)

    plt.bar(dates, confirmed, color='lightblue',
        label="Nombre total de cas confirmés")
    plt.bar(dates, deaths, color='red',
        label="Nombre total de décès signalés")
    plt.xlabel("Dates")
    plt.ylabel("Nombre de personnes")
    plt.title(country)
    plt.legend()
    plt.show()

## Diagrammes circulaires

In [ ]:
def plot_pie(df, column, title):
    labels = list(df.index)
    sizes = list(df[column])
    explode = [0] * len(labels)

    with plt.style.context(
        { "axes.prop_cycle" : plt.cycler("color",
        plt.cm.tab20c.colors)}):
        fig1, ax1 = plt.subplots()
        ax1.pie(sizes,
                explode=explode,
                labels=labels,
                autopct='%1.2f%%',
                shadow=False,
                startangle=0)
        ax1.axis('equal')  # Un rapport d'aspect égal garantit que le diagramme
                           # sera dessiné en cercle.
        plt.legend()
        plt.title(title)

    centre_circle = plt.Circle((0,0),0.70,fc='white')
    fig = plt.gcf()
    fig.gca().add_artist(centre_circle)
    plt.show()

In [ ]:
top10_countries_confirmed = df_most_confirmed_recent_sorted.head(10)
plot_pie(top10_countries_confirmed,
         "Confirmed",
         "Top 10 Pays pour les cas confirmés")

In [ ]:
plot_pie(df_combined_china_most_recent_sorted,
         "Confirmed",
         "Top 10 provinces chinoises pour les cas confirmés")

In [ ]:
df_most_deaths_recent_sorted = df_most_deaths_recent_sorted.head(10)
plot_pie(df_most_deaths_recent_sorted,
         "Deaths",
         "Top 10 des pays pour les décès")

## Aplatir les courbes : nouveaux cas par jour

In [ ]:
plt.rcParams["figure.figsize"] = (12, 7)

# pays à tracer
pays = ["Afghanistan", "New Zealand", "South Africa", "Spain"]

# pour chaque pays
for country, df_country in df_daily.groupby(level=1):
    if country not in pays:
        continue

    # obtenir la liste des dates
    dates = list(df_country.index.get_level_values('Date'))

    # dérouler (inverse de la somme cumulée) les cas confirmés et les décès
    daily = df_country[['Confirmed', 'Deaths']]
    df_unrolled = daily.diff().fillna(daily)

    # tracer les graphiques linéaires
    plt.plot(dates, df_unrolled['Confirmed'], color='blue',
             label="Cas confirmés par jour")
    plt.plot(dates, df_unrolled['Deaths'], color='red',
             label="Décès par jour")
    plt.xlabel("Dates")
    plt.ylabel("Nombre de personnes")
    plt.title(country)
    plt.legend()
    plt.show()

## Carte du monde avec Folium

Sur Google Colab, Folium est déjà installé. Sinon : `pip install folium`.

In [ ]:
import folium

# obtenir la date la plus récente du DataFrame
most_recent_date = df_combined['Date'].max()

# obtenir toutes les données de cette date la plus récente
df_latest_day = df_combined[df_combined.Date == most_recent_date]

# créer une carte
folium_map = folium.Map(location=[40.738, -73.98], zoom_start=4)
scale = 25

for _, row in df_latest_day.iterrows():
    lat = row["Lat"]
    lng = row["Long"]
    confirmed = row["Confirmed"]
    deaths = row["Deaths"]

    # Cas confirmés : cercle sur la province/état/région/pays
    marker_confirmed = folium.CircleMarker(location=[lat, lng],
                                           radius=np.sqrt(confirmed) / scale,
                                           color="green",
                                           fill=True)
    marker_confirmed.add_to(folium_map)

    # Décès : cercle sur la province/état/région/pays
    marker_deaths = folium.CircleMarker(location=[lat, lng],
                                        radius=np.sqrt(deaths) / scale,
                                        color="red",
                                        fill=True)
    marker_deaths.add_to(folium_map)

folium_map.save("Covid-19 Map.html")

Dans un notebook, affiche la carte en évaluant la variable :

In [ ]:
folium_map